# Homework 5: UFO Sightings Visualization

In this assignment, I explore UFO sighting reports using Python, Pandas,
and Altair. I will create two visualizations to examine how UFO sightings
have changed over time and how reported UFO shapes differ across countries.

The dataset is provided by the IS 445 course repository.

- source: https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv 

# 1. Import Libraries

In [1]:
import pandas as pd
import altair as alt

# 2. Load Dataset

In [2]:
# Load the UFO sightings dataset
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

df = pd.read_csv(url)

print("*Dataset shape:", df.shape)
df.head()

*Dataset shape: (80331, 11)


,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 1949-50. It occurred after a Boy Scout meeting in the Baptist Church. The Baptist Church sit,4/27/2004,29.8830556,-97.9411111
0,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
1,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
2,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
3,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611
4,10/10/1961 19:00,bristol,tn,us,sphere,300.0,5 minutes,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889


### After loading the dataset, I noticed that the first row was incorrectly used as column headers, so I manually assigned appropriate column names.

In [3]:
# Reload the dataset without treating the first row as headers
df = pd.read_csv(url, header=None, low_memory=False)   #Referreed to: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html

# Assign appropriate column names
df.columns = [
    'datetime', 'city', 'state', 'country', 'shape',
    'duration_seconds', 'duration_hours_min', 'comments',
    'date_posted', 'latitude', 'longitude'
]

df.head()

,datetime,city,state,country,shape,duration_seconds,duration_hours_min,comments,date_posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


# 3. Data Preparation

To examine how UFO sightings have changed over time, I converted the `datetime` column into a datetime format and extracted the year. I then grouped the observations by year to calculate the total number of reported sightings.

In [4]:
# Convert datetime strings to datetime objects
df['datetime'] = pd.to_datetime(     #Referred to: https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
    df['datetime'],
    errors='coerce',
    format='mixed'
)

# Extract the year
df['year'] = df['datetime'].dt.year

# Check missing values and the range of years
print("*Missing years:", df['year'].isna().sum())
print("*Year range:", df['year'].min(), "-", df['year'].max())

*Missing years: 0
*Year range: 1906 - 2014


In [5]:
# Count sightings by year
df_yearly = (
    df.dropna(subset=['year'])
      .groupby('year')
      .size()
      .reset_index(name='sightings')
)

df_yearly['year'] = df_yearly['year'].astype(int)

df_yearly.head()

,year,sightings
0,1906,1
1,1910,2
2,1916,1
3,1920,1
4,1925,1


# 4. Visualization

## 4-1. Visualization 1: UFO Sightings Over Time

This visualization shows the number of reported UFO sightings by year. I used a line chart to examine the overall trend over time, with tooltips to display the exact year and number of sightings.

In [6]:
# Line chart of yearly UFO sightings
chart1 = alt.Chart(df_yearly).mark_line().encode(      #Referred to: https://altair-viz.github.io/gallery/simple_line_chart.html & https://altair-viz.github.io/user_guide/customization.html
    x=alt.X('year:Q', title='Year'),
    y=alt.Y('sightings:Q', title='Number of Sightings'),
    tooltip=['year:Q', 'sightings:Q'],
    size=alt.value(4)   # Slightly increased the line thickness to make hovering easier   
).properties(
    title='UFO Sightings Over Time',
    width=600,
    height=350
)

display(chart1)

alt.Chart(...)

## 4-2. Visualization 2: UFO Sightings by Shape and Country

For the second visualization, I wanted to **compare the number of UFO sightings by shape across different countries.** 

I removed rows with missing country or shape values and used `groupby()` and `size()` to count sightings for each country and shape. I then created a bar chart and added a dropdown menu to allow users to select a country and explore its distribution of UFO shapes.

In [7]:
# Count UFO sightings by country and shape
df_shapes = df.dropna(subset=['country', 'shape'])

df_shapes = df_shapes.groupby(['country', 'shape']).size().reset_index(name='sightings')  #Referred to: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html

df_shapes.head()

,country,shape,sightings
0,au,changing,9
1,au,chevron,3
2,au,cigar,15
3,au,circle,62
4,au,cone,6


In [8]:
# Bar chart for UFO shapes in the US

# Filter UFO sightings reported in the US
us = df_shapes[df_shapes['country'] == 'us']

chart2_basic = alt.Chart(us).mark_bar().encode(             #Referred to: https://altair-viz.github.io/user_guide/marks/bar.html
    x=alt.X('shape:N', sort='-y', title='UFO Shape'),
    y=alt.Y('sightings:Q', title='Number of Sightings')
).properties(
    title='UFO Sightings by Shape in the US',
    width=600,
    height=350
)

display(chart2_basic)

alt.Chart(...)

## 4-3. Adding Interactivity

To make the second visualization interactive, I added a **dropdown menu** that allows users to select a country. The bar chart updates to show the number of UFO sightings by shape for the selected country.

In [9]:
# Create a country dropdown
countries = sorted(df_shapes['country'].unique().tolist())

dropdown = alt.binding_select(              #Referred to: https://altair-viz.github.io/gallery/multiple_interactions.html
    options=['us', 'gb', 'de', 'ca', 'au'],
    labels=['United States', 'United Kingdom',
            'Germany', 'Canada', 'Australia'],
    name='Country: '
)   

selection = alt.selection_point(
    fields=['country'],
    bind=dropdown,
    value='us'
)

In [10]:
# Interactive horizontal bar chart
chart2 = alt.Chart(df_shapes).mark_bar().encode(
    x=alt.X('sightings:Q', title='Number of Sightings'),
    y=alt.Y('shape:N', sort='-x', title='UFO Shape'),
    tooltip=['shape:N', 'sightings:Q']
).add_params(
    selection
).transform_filter(
    selection
).properties(
    title='UFO Sightings by Shape and Country',
    width=600,
    height=450
)

display(chart2)

alt.Chart(...)

# 5. Exporting Visualizations

I saved both Altair charts as JSON files so they can be embedded in my Jekyll website.

In [11]:
# Save both charts as Vega-Lite JSON files
chart1.save('ufo_trend.json')     #Referred to: https://altair-viz.github.io/user_guide/saving_charts.html 
chart2.save('ufo_shapes.json')